<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/wip-cartpole-dqn-lightning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CartPole-v1 - DQN with PyTorch Lightning

This notebook trains a **Deep Q-Network (DQN)** agent on the classic CartPole-v1 environment using **PyTorch Lightning**.

In [ ]:
!pip install gymnasium pytorch-lightning wandb tsilva-notebook-utils

In [ ]:

import random, math
import numpy as np
import gymnasium as gym
import torch
import torch.nn as nn
import pytorch_lightning as pl
from collections import deque
from tsilva_notebook_utils.video import render_video

CONFIG = {
    'gamma': 0.99,
    'batch_size': 64,
    'replay_size': 10000,
    'lr': 1e-3,
    'eps_start': 1.0,
    'eps_end': 0.05,
    'eps_decay': 500,
    'target_update': 10,
    'max_steps': 5000,
    'seed': 42,
}
pl.seed_everything(CONFIG['seed'], workers=True)


In [ ]:

class ReplayBuffer:
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)
    def push(self, *transition):
        self.buffer.append(transition)
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        return map(np.stack, zip(*batch))
    def __len__(self):
        return len(self.buffer)


In [ ]:

class QNetwork(nn.Module):
    def __init__(self, obs_dim, n_actions):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, 128),
            nn.ReLU(),
            nn.Linear(128, n_actions)
        )
    def forward(self, x):
        return self.net(x)

class LightningDQN(pl.LightningModule):
    def __init__(self, obs_dim, n_actions, config):
        super().__init__()
        self.save_hyperparameters(ignore=['config'])
        self.config = config
        self.q_net = QNetwork(obs_dim, n_actions)
        self.target_net = QNetwork(obs_dim, n_actions)
        self.target_net.load_state_dict(self.q_net.state_dict())
        self.env = gym.make('CartPole-v1')
        self.buffer = ReplayBuffer(config['replay_size'])
        self.state, _ = self.env.reset(seed=config['seed'])
        self.total_steps = 0

    def forward(self, x):
        return self.q_net(x)

    def act(self, state):
        eps = self.config['eps_end'] + (self.config['eps_start'] - self.config['eps_end']) * math.exp(-1.0 * self.total_steps / self.config['eps_decay'])
        if random.random() < eps:
            return self.env.action_space.sample()
        else:
            state = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
            with torch.no_grad():
                q = self.q_net(state)
            return int(torch.argmax(q, dim=1)[0].item())

    def train_dataloader(self):
        class _Infinite(torch.utils.data.IterableDataset):
            def __iter__(self):
                while True:
                    yield 0
        return torch.utils.data.DataLoader(_Infinite(), batch_size=1)

    def training_step(self, batch, batch_idx):
        self.total_steps += 1
        action = self.act(self.state)
        next_state, reward, terminated, truncated, _ = self.env.step(action)
        done = terminated or truncated
        self.buffer.push(self.state, action, reward, next_state, done)
        self.state = next_state if not done else self.env.reset()[0]

        if len(self.buffer) < self.config['batch_size']:
            return None

        states, actions, rewards, next_states, dones = self.buffer.sample(self.config['batch_size'])
        states = torch.tensor(states, dtype=torch.float32, device=self.device)
        actions = torch.tensor(actions, dtype=torch.long, device=self.device).unsqueeze(-1)
        rewards = torch.tensor(rewards, dtype=torch.float32, device=self.device)
        next_states = torch.tensor(next_states, dtype=torch.float32, device=self.device)
        dones = torch.tensor(dones, dtype=torch.float32, device=self.device)

        q_values = self.q_net(states).gather(1, actions).squeeze()
        next_q = self.target_net(next_states).max(1)[0]
        targets = rewards + self.config['gamma'] * next_q * (1 - dones)
        loss = nn.functional.mse_loss(q_values, targets.detach())
        self.log('loss', loss, on_step=True, prog_bar=True)

        if self.global_step % self.config['target_update'] == 0:
            self.target_net.load_state_dict(self.q_net.state_dict())
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.q_net.parameters(), lr=self.config['lr'])


In [ ]:

model = LightningDQN(4, 2, CONFIG)
trainer = pl.Trainer(max_steps=CONFIG['max_steps'], log_every_n_steps=50, enable_model_summary=False)
trainer.fit(model)


In [ ]:

def play(model, n_episodes=1):
    env = gym.make('CartPole-v1', render_mode='rgb_array')
    frames = []
    for _ in range(n_episodes):
        state, _ = env.reset(seed=CONFIG['seed'])
        done = False
        while not done:
            frames.append(env.render())
            state_t = torch.tensor(state, dtype=torch.float32, device=model.device).unsqueeze(0)
            with torch.no_grad():
                action = torch.argmax(model.q_net(state_t), dim=1).item()
            next_state, _, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            state = next_state
    env.close()
    return render_video(frames)

play(model)
